# Когортный анализ на данных Brazilian E-Commerce

1. Импорт библиотек, загрузка и проверка данных

In [2]:
import pandas as pd

orders = pd.read_csv('C:/Users/aablyazizov/Desktop/DS/Mentoring/Brazilian E-Commerce/data/olist_orders_dataset.csv')
payments = pd.read_csv('C:/Users/aablyazizov/Desktop/DS/Mentoring/Brazilian E-Commerce/data/olist_order_payments_dataset.csv')
customers = pd.read_csv('C:/Users/aablyazizov/Desktop/DS/Mentoring/Brazilian E-Commerce/data/olist_customers_dataset.csv')

print(orders.shape)
print(payments.shape)
print(customers.shape)

(99441, 8)
(103886, 5)
(99441, 5)


In [3]:
print(orders.columns.tolist())
print(payments.columns.tolist())
print(customers.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


2. Объединяем 2 набора данных (присоединяем платежи к заказам)

In [5]:
df = pd.merge(orders, payments, on='order_id', how='left')

In [6]:
df_delivered = df[df['order_status'] == 'delivered']

3. Рассчитываем arppu

In [9]:
arppu = round(df_delivered['payment_value'].sum()/df_delivered['customer_id'].nunique(), 2)
arppu

159.85

4. Создадим атрибут, хранящий месяц и год первого платежа клиента и соединим с датафреймом заказов

In [10]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.to_period('M')

In [11]:
first_purchase = orders.groupby('customer_id')['purchase_month'].min().reset_index()
first_purchase.columns = ['customer_id', 'cohort_month']

In [12]:
orders = pd.merge(orders, first_purchase, on='customer_id', how='left')

In [13]:
orders[['customer_id', 'purchase_month', 'cohort_month']].head(10)

,customer_id,purchase_month,cohort_month
0,9ef432eb6251297304e76186b10a928d,2017-10,2017-10
1,b0830fb4747a6c6d20dea0b8c802d7ef,2018-07,2018-07
2,41ce2a54c0b03bf3443c3d931a367089,2018-08,2018-08
3,f88197465ea7920adcdbec7375364d82,2017-11,2017-11
4,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02,2018-02
5,503740e9ca751ccdda7ba28e9ab8f608,2017-07,2017-07
6,ed0271e0b7da060a393796590e7b737a,2017-04,2017-04
7,9bdf08b4b3b52b5526ff42d37d47f222,2017-05,2017-05
8,f54a9f0e6b351c431402b8461ea51999,2017-01,2017-01
9,31ad1d1b63eb9962463f764d4e6e0c9d,2017-07,2017-07


5. Присоединим клиентов к датафрейму заказов

In [14]:
orders = pd.merge(orders, customers, on = 'customer_id', how = 'left')

In [15]:
orders.columns.tolist()

['order_id',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'purchase_month',
 'cohort_month',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state']

6. Создадим атрибуты когорт и посмотрим в процентном соотношении сколько клиентов возвращаются через разное кол-во месяцев для разных когорт

In [16]:
first_unique_order_date = orders.groupby('customer_unique_id')['purchase_month'].min().reset_index()

In [17]:
first_unique_order_date.columns = ['customer_unique_id', 'cohort_month']

In [18]:
orders = pd.merge(orders, first_unique_order_date, on = 'customer_unique_id', how = 'left')

In [19]:
orders = orders.drop(columns=['cohort_month_y'], errors='ignore')
orders = orders.rename(columns={'cohort_month_x': 'cohort_month'})

In [20]:
orders['cohort_index'] = (12 * (orders['purchase_month'].dt.year - orders['cohort_month'].dt.year) +
    (orders['purchase_month'].dt.month - orders['cohort_month'].dt.month))

In [21]:
cohort_data = orders.groupby(['cohort_month', 'cohort_index'])['customer_unique_id'].nunique().reset_index()

In [22]:
cohort_pivot = cohort_data.pivot_table(index='cohort_month', columns='cohort_index', values='customer_unique_id')

In [23]:
cohort_size = cohort_pivot[0]
retention = cohort_pivot.divide(cohort_size, axis=0).round(3) * 100

print(retention.iloc[:8, :6])

cohort_index      0
cohort_month       
2016-09       100.0
2016-10       100.0
2016-12       100.0
2017-01       100.0
2017-02       100.0
2017-03       100.0
2017-04       100.0
2017-05       100.0


In [24]:
repeat_buyers = orders.groupby('customer_unique_id')['order_id'].count()
print(repeat_buyers[repeat_buyers > 1].count())
print(repeat_buyers.value_counts().head())

2997
order_id
1    93099
2     2745
3      203
4       30
5        8
Name: count, dtype: int64


In [25]:
print(orders['cohort_index'].value_counts().sort_index().head(10))

cohort_index
0    99441
Name: count, dtype: int64


In [26]:
repeat = orders.groupby('customer_unique_id').filter(lambda x: len(x) > 1)
print(repeat[['customer_unique_id', 'purchase_month', 'cohort_month', 'cohort_index']].head(10))

                  customer_unique_id purchase_month cohort_month  cohort_index
0   7c396fd4830fd04220f754e42b4e5bff        2017-10      2017-10             0
15  ccafc1c3f270410521c3c6f3b249870f        2018-06      2018-06             0
16  6e26bbeaa107ec34112c64e1ee31c0f5        2018-01      2018-01             0
44  08fb46d35bb3ab4037202c23592d1259        2018-06      2018-06             0
46  c2551ea089b7ebbc67a2ea8757152514        2017-05      2017-05             0
58  51838d41add414a0b1b989b7d251d9ee        2017-03      2017-03             0
72  fa0ee7ceb94193fb02aa78ce3a55695a        2018-02      2018-02             0
79  bb4d84a2b45b22ed710ac8c0dec63d1a        2017-07      2017-07             0
80  2ae3c67452283d5a0d30b32e0d33296e        2017-12      2017-12             0
82  04e495a3f45df8b41be2e934bbc16961        2017-05      2017-05             0


Видим, что для данного датасета не обнаружены возвращения клиентов. Делаем вывод, что 